In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.10: Contractive VAE (Smooth + Probabilistic)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Combine VAE's probabilistic latent space with Contractive AE's smoothness
#   - Compare latent interpolation: vanilla VAE vs Contractive VAE
#   - Understand WHY smooth + generative is the best of both worlds
#   - Apply to drug molecule similarity search at a Singapore biotech
#   - Build a latent-space nearest-neighbour system for molecular similarity
#
# PREREQUISITES: 09_vae.py
# ESTIMATED TIME: ~20 min
#
# TASKS:
#   1. Build Contractive VAE (VAE + Jacobian weight penalty)
#   2. Train and compare interpolation smoothness vs vanilla VAE
#   3. Visualise side-by-side interpolation comparison
#   4. Apply: molecular feature similarity search
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.spatial.distance import cdist



## THEORY — Smooth + Generative = Best of Both Worlds


In [ ]:
# The Contractive VAE combines:
#   - VAE's probabilistic latent space (can sample new data)
#   - Contractive AE's Jacobian penalty (smooth transitions)
#
# The result: a latent space where you can BOTH generate new samples
# AND navigate smoothly between them. Small steps in latent space
# produce small, meaningful changes in output.
#
# WHY THIS MATTERS: In drug discovery, you want to explore the space
# of possible molecules smoothly. A drug that works for condition A
# might have a close neighbour in latent space that works for condition
# B. The smooth latent space means "close in latent space" reliably
# means "structurally similar as molecules" — no sharp discontinuities
# where a tiny latent change produces a completely different molecule.



## TASK 1 — Load data, engines, and train vanilla VAE for comparison


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()


# --- Vanilla VAE (for comparison) ---
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decoder(z), mu, logvar

    def sample(self, n):
        z = torch.randn(
            n, self.fc_mu.out_features, device=next(self.parameters()).device
        )
        return self.decoder(z)


KL_WEIGHT = 0.1


def vae_loss_fn(model, xb):
    x_hat, mu, logvar = model(xb)
    recon = F.mse_loss(x_hat, xb, reduction="sum") / xb.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / xb.size(0)
    return recon + KL_WEIGHT * kl, {}


print("Training vanilla VAE for comparison...")
vae_model = VAE(INPUT_DIM, LATENT_DIM)
vae_losses = train_variant(
    tracker,
    exp_name,
    vae_model,
    "vae_compare",
    flat_loader,
    vae_loss_fn,
    extra_params={"kl_weight": str(KL_WEIGHT)},
)



## TASK 2 — Build and Train Contractive VAE


In [ ]:
class ContractiveVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Define explicit encoder layers (for weight access):
        #       enc1: Linear(input_dim, 256)
        #       enc2: Linear(256, 128)
        #       fc_mu: Linear(128, latent_dim)
        #       fc_logvar: Linear(128, latent_dim)
        self.enc1 = ____
        self.enc2 = ____
        self.fc_mu = ____
        self.fc_logvar = ____

        # TODO: Build decoder — same as vanilla VAE
        self.decoder = ____

    def encoder(self, x):
        # TODO: Forward through enc1->ReLU->enc2->ReLU
        ____

    def encode(self, x):
        # TODO: encoder(x) -> (mu, logvar)
        ____

    def reparameterise(self, mu, logvar):
        # TODO: Reparameterisation trick
        ____

    def forward(self, x):
        # TODO: encode -> reparameterise -> decode
        # Return (reconstruction, mu, logvar)
        ____

    def sample(self, n):
        # TODO: Sample z ~ N(0,I), decode
        ____


CVAE_CONTRACTIVE_WEIGHT = 1e-4


def cvae_loss_fn(model, xb):
    # TODO: ELBO loss + Jacobian penalty on enc1.weight and enc2.weight
    # recon + KL_WEIGHT * kl + CVAE_CONTRACTIVE_WEIGHT * jacobian_penalty
    ____


print("\n" + "=" * 70)
print("  Contractive VAE — Smooth + Probabilistic")
print("=" * 70)

# TODO: Create ContractiveVAE(INPUT_DIM, LATENT_DIM) and train
cvae_model = ____
cvae_losses = ____

# TODO: show_reconstruction
____



## TASK 3 — Compare Interpolation Smoothness: VAE vs CVAE


In [ ]:
def compare_interpolation(model_a, model_b, test_data, label_a, label_b, n_steps=10):
    # TODO: Create 2-row interpolation comparison figure
    # Row 0: model_a interpolation from image 0 to image 5
    # Row 1: model_b interpolation from image 0 to image 5
    # For each model: encode start/end -> interpolate in latent space -> decode
    # Save to OUTPUT_DIR / "ex1_vae_vs_cvae_interpolation.png"
    fig, axes = plt.subplots(2, n_steps + 2, figsize=(16, 3.5))
    ____
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_vs_cvae_interpolation.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


compare_interpolation(
    vae_model, cvae_model, X_test_flat, "Vanilla VAE", "Contractive VAE"
)



### Checkpoint


In [ ]:
assert len(cvae_losses) == EPOCHS
assert cvae_losses[-1] < cvae_losses[0]
print("\n--- Checkpoint passed --- contractive VAE trained\n")

if has_registry:
    register_model(registry, "contractive_vae", cvae_model, cvae_losses[-1])



## APPLY — Drug Molecule Similarity Search


In [ ]:
# BUSINESS SCENARIO: You are an ML engineer at a Singapore biotech
# company (A*STAR spinoff). Drug discovery involves exploring vast
# molecular spaces. Given a lead compound that shows promise, you
# want to find structurally similar molecules that might have improved
# properties. The CVAE's smooth latent space means "nearby in latent
# space" = "structurally similar as molecules."

print("\n" + "=" * 70)
print("  APPLICATION: Molecular Similarity Search")
print("=" * 70)

# --- Generate synthetic molecular descriptor data ---
N_MOLECULES = 5000
N_DESCRIPTORS = 50
N_FAMILIES = 5
FAMILY_NAMES = ["Analgesics", "Antibiotics", "Antivirals", "Oncology", "Cardiovascular"]

mol_rng = np.random.default_rng(42)

# TODO: Generate clustered molecular data
# family_labels = random assignment to N_FAMILIES
# family_centers = random centers in N_DESCRIPTORS-dim space
# mol_data[i] = family_centers[label[i]] + noise
# Normalise to [0, 1]
family_labels = ____
family_centers = ____
mol_data = ____

mol_min, mol_max = mol_data.min(axis=0), mol_data.max(axis=0)
mol_range = mol_max - mol_min
mol_range[mol_range == 0] = 1.0
mol_norm = (mol_data - mol_min) / mol_range

mol_tensor = torch.tensor(mol_norm, device=device)
mol_loader = DataLoader(TensorDataset(mol_tensor), batch_size=128, shuffle=True)


class MolecularCVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=10):
        super().__init__()
        # TODO: enc1: Linear(input_dim, 64)
        #       enc2: Linear(64, 32)
        #       fc_mu: Linear(32, latent_dim)
        #       fc_logvar: Linear(32, latent_dim)
        #       decoder: Linear(latent_dim, 32), ReLU, Linear(32, 64), ReLU,
        #                Linear(64, input_dim), Sigmoid
        self.enc1 = ____
        self.enc2 = ____
        self.fc_mu = ____
        self.fc_logvar = ____
        self.decoder = ____

    def encode(self, x):
        # TODO: enc1->ReLU->enc2->ReLU->(mu, logvar)
        ____

    def forward(self, x):
        # TODO: encode -> reparameterise -> decode
        ____


MOL_LATENT = 10
# TODO: Create MolecularCVAE, optimizer. Train 80 epochs with ELBO + Jacobian loss.
mol_model = ____
mol_opt = ____

print(f"Training CVAE on {N_MOLECULES} molecules ({N_DESCRIPTORS} descriptors)...")
for epoch in range(80):
    mol_model.train()
    for (batch,) in mol_loader:
        # TODO: Forward, compute ELBO + contractive loss, backprop
        ____

# --- Encode all molecules ---
mol_model.eval()
with torch.no_grad():
    all_mu, _ = mol_model.encode(mol_tensor)
    mol_latents = all_mu.cpu().numpy()

# --- Similarity search ---
query_idx = 42
query_latent = mol_latents[query_idx : query_idx + 1]
dists = cdist(query_latent, mol_latents, metric="euclidean")[0]
dists[query_idx] = float("inf")
top_k = 10
nearest = np.argsort(dists)[:top_k]

query_family = family_labels[query_idx]
retrieved_families = family_labels[nearest]
same_family_pct = (retrieved_families == query_family).mean() * 100

print(f"\nQuery molecule #{query_idx} (family: {FAMILY_NAMES[query_family]})")
print(f"Top-{top_k} nearest neighbours:")
for rank, ni in enumerate(nearest):
    fam = FAMILY_NAMES[family_labels[ni]]
    match = "SAME" if family_labels[ni] == query_family else "diff"
    print(f"  #{rank+1}: molecule {ni}, family={fam}, dist={dists[ni]:.3f} [{match}]")
print(f"Same-family retrieval: {same_family_pct:.0f}%")

# --- Visualise latent space (PCA) ---
from numpy.linalg import svd

# TODO: PCA to 2D, scatter plot coloured by drug family
# Save to OUTPUT_DIR / "ex1_molecular_latent_space.png"
centered = mol_latents - mol_latents.mean(axis=0)
_, _, Vt = svd(centered, full_matrices=False)
proj = centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(10, 8))
____
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ex1_molecular_latent_space.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Business Impact ---
print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Drug Molecule Similarity Search")
print("=" * 64)
print(f"\nMolecular library: {N_MOLECULES:,} compounds, {N_DESCRIPTORS} descriptors")
print(f"CVAE latent dimension: {MOL_LATENT}")
print(f"Same-family retrieval rate: {same_family_pct:.0f}% (top-{top_k})")
print(f"\nDrug discovery impact:")
print(f"  Traditional screening: test 10,000 compounds at S$100 each = S$1M")
print(f"  CVAE-guided search: prioritise top-100 neighbours first")
print(f"  Expected hit rate improvement: ~3-5x (from random screening)")
print(f"  Cost savings per drug programme: S$200K-400K in early screening")
print("=" * 64)



## REFLECTION


[x] Built a Contractive VAE (VAE + Jacobian weight penalty)
  [x] Compared interpolation smoothness: vanilla VAE vs CVAE
  [x] Observed smoother transitions in CVAE's latent space
  [x] Applied to molecular similarity search in drug discovery
  [x] Built latent-space nearest-neighbour retrieval for molecules
  [x] Quantified drug screening cost savings

  KEY INSIGHT: The CVAE is the best of both worlds. The VAE component
  gives you a generative model (sample new molecules). The contractive
  component gives you a smooth latent space (nearby molecules are truly
  similar). Together, they enable both GENERATION (propose new drug
  candidates) and NAVIGATION (find similar molecules, interpolate
  between hits and misses).

  Next: 11_grand_comparison.py puts all 10 variants side by side...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (VAE + Jacobian penalty)


In [ ]:
# Contractive VAE = VAE loss + Jacobian penalty. Expect the BLOOD
# TEST to show low-but-stable gradients — both regularisers pull
# the encoder toward smoother manifolds.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = cvae_loss_fn(m, xb)
    return loss


print("\n── Diagnostic Report (Contractive VAE) ──")
diag, findings = run_diagnostic_checkpoint(
    cvae_model,
    flat_loader,
    _diag_loss,
    title=f"CVAE (KL={KL_WEIGHT}, lam={CVAE_CONTRACTIVE_WEIGHT})",
    n_batches=8,
    train_losses=cvae_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [!] Gradient flow (WARNING): Compound dampening at
#       'fc_mu.weight' — RMS = 3.1e-05. Both KL penalty AND
#       Jacobian penalty act on mu-head. Contrast: VAE
#       (09) had 5.2e-05, ContractiveAE (05) had 7.4e-05 —
#       CVAE sits below both because TWO regularisers.
#   [!] Dead neurons  (WARNING): 'decoder.2' (relu): 18%
#       dead — early-epoch VAE signature plus contractive
#       dampening of encoder gradients.
#   [✓] Loss trend    (HEALTHY): 3-term composition, all
#       decreasing. Total slope -2.3e-03/epoch. Final loss
#       ~0.046 (recon 0.033 + KL 0.009 + Jacobian 0.004).



## Final train loss: ~0.046, 10 epochs, beta=1, lambda=1e-4.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — TWO REGULARISERS COMPOUNDING] RMS 3.1e-05
    at fc_mu is the key diagnostic. This is the SUM of
    two dampening effects: (a) KL penalty shrinking encoder
    gradients (see 09 VAE), (b) Jacobian penalty further
    constraining encoder sensitivity (see 05 contractive).
    The floor sits BELOW either alone. If it drops below
    1e-5, one regulariser is winning — halve the heavier
    weight first.
    >> Prescription: Compute the ratio
       beta*KL_loss : lambda*Jacobian_loss. If >3:1 or
       <1:3, one is dominating. Target 1:1 to 2:1 (KL
       primary, Jacobian secondary).

 [X-RAY] 18% dead is the VAE early-epoch pattern persisting
    slightly longer due to contractive dampening (slower
    decoder adaptation). Should converge to <10% by epoch 8.
    If it stays above 15%, EITHER the KL is too strong
    (posterior collapse risk) OR Jacobian is too strong
    (encoder stuck). Diagnose by reading the Blood Test
    first: which weight is dampening more?
    >> Prescription: If KL term dominates: anneal beta.
       If Jacobian dominates: halve lambda.

 [STETHOSCOPE — THREE-TERM COMPOSITION] Unlike 09 (2 terms),
    CVAE's loss is recon + KL + Jacobian. diag.epochs_df()
    should expose all three. A healthy run sees: recon falls
    fastest (decoder learns), KL falls next (latent tightens),
    Jacobian last (encoder smooths). Any other ordering
    signals imbalance — e.g. Jacobian dropping first means
    lambda is too large and is crushing learning.
    >> Prescription: Read three curves. If Jacobian curve
       is flat, lambda is too small — no contraction
       benefit. If Jacobian drops faster than recon,
       lambda is too large — crushing reconstruction.

 FIVE-INSTRUMENT TAKEAWAY: CVAE demonstrates the DIAGNOSTIC
 SKILL OF DISENTANGLING MULTIPLE REGULARISERS. Same Blood
 Test signal (low RMS), but the attribution depends on
 weighing the terms. This reading skill scales to ex_5 GANs
 (generator regulariser + discriminator regulariser +
 gradient penalty = 3 terms to balance) and to ex_8 RL
 (policy + value + entropy bonuses). Clinical reading in
 the face of multiple compounding signals is the ceiling
 skill for this module.


In [ ]:
show_reconstruction(cvae_model, X_test_flat, "Contractive VAE")



## TASK 3 — Compare Interpolation Smoothness: VAE vs CVAE


In [ ]:
def compare_interpolation(model_a, model_b, test_data, label_a, label_b, n_steps=10):
    fig, axes = plt.subplots(2, n_steps + 2, figsize=(16, 3.5))
    fig.suptitle(
        f"Latent Interpolation: {label_a} vs {label_b}", fontsize=13, fontweight="bold"
    )

    for row, (model, label) in enumerate([(model_a, label_a), (model_b, label_b)]):
        model.eval()
        with torch.no_grad():
            x1, x2 = test_data[0:1].to(device), test_data[5:6].to(device)
            if hasattr(model, "encode"):
                mu1, _ = model.encode(x1)
                mu2, _ = model.encode(x2)
                z1, z2 = mu1, mu2
            else:
                z1, z2 = model.encoder(x1), model.encoder(x2)
            alphas = torch.linspace(0, 1, n_steps).to(device)

            axes[row, 0].imshow(x1[0].cpu().reshape(28, 28), cmap="gray")
            axes[row, 0].axis("off")
            if row == 0:
                axes[row, 0].set_title("Start", fontsize=7)

            for i, alpha in enumerate(alphas):
                z = (1 - alpha) * z1 + alpha * z2
                x_hat = model.decoder(z)
                axes[row, i + 1].imshow(
                    x_hat[0].cpu().reshape(28, 28).clamp(0, 1), cmap="gray"
                )
                axes[row, i + 1].axis("off")
                if row == 0:
                    axes[row, i + 1].set_title(f"{alpha:.1f}", fontsize=7)

            axes[row, -1].imshow(x2[0].cpu().reshape(28, 28), cmap="gray")
            axes[row, -1].axis("off")
            if row == 0:
                axes[row, -1].set_title("End", fontsize=7)

        axes[row, 0].set_ylabel(label, fontsize=9, rotation=0, labelpad=50)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_vs_cvae_interpolation.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


compare_interpolation(
    vae_model, cvae_model, X_test_flat, "Vanilla VAE", "Contractive VAE"
)



### Checkpoint


In [ ]:
assert len(cvae_losses) == EPOCHS
assert cvae_losses[-1] < cvae_losses[0]
print("\n--- Checkpoint passed --- contractive VAE trained\n")

if has_registry:
    register_model(registry, "contractive_vae", cvae_model, cvae_losses[-1])



## APPLY — Drug Molecule Similarity Search


In [ ]:
# BUSINESS SCENARIO: You are an ML engineer at a Singapore biotech
# company (A*STAR spinoff). Drug discovery involves exploring vast
# molecular spaces. Given a lead compound that shows promise, you
# want to find structurally similar molecules that might have improved
# properties. The CVAE's smooth latent space means "nearby in latent
# space" = "structurally similar as molecules."
#
# We simulate this with Fashion-MNIST: each image class represents
# a "molecular family." Smooth interpolation between families suggests
# the latent space can guide molecular optimisation.

print("\n" + "=" * 70)
print("  APPLICATION: Molecular Similarity Search")
print("=" * 70)

# --- Generate synthetic molecular descriptor data ---
N_MOLECULES = 5000
N_DESCRIPTORS = 50  # molecular descriptors (LogP, MW, TPSA, etc.)
N_FAMILIES = 5  # drug families (analgesics, antibiotics, etc.)
FAMILY_NAMES = ["Analgesics", "Antibiotics", "Antivirals", "Oncology", "Cardiovascular"]

mol_rng = np.random.default_rng(42)

# Generate clustered molecular data
family_labels = mol_rng.choice(N_FAMILIES, size=N_MOLECULES)
family_centers = (
    mol_rng.standard_normal((N_FAMILIES, N_DESCRIPTORS)).astype(np.float32) * 2
)
mol_data = np.zeros((N_MOLECULES, N_DESCRIPTORS), dtype=np.float32)
for i in range(N_MOLECULES):
    mol_data[i] = family_centers[family_labels[i]] + mol_rng.normal(
        0, 0.5, N_DESCRIPTORS
    ).astype(np.float32)

# Normalise
mol_min, mol_max = mol_data.min(axis=0), mol_data.max(axis=0)
mol_range = mol_max - mol_min
mol_range[mol_range == 0] = 1.0
mol_norm = (mol_data - mol_min) / mol_range

# Train CVAE on molecular data
mol_tensor = torch.tensor(mol_norm, device=device)
mol_loader = DataLoader(TensorDataset(mol_tensor), batch_size=128, shuffle=True)


class MolecularCVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=10):
        super().__init__()
        self.enc1 = nn.Linear(input_dim, 64)
        self.enc2 = nn.Linear(64, 32)
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = F.relu(self.enc1(x))
        h = F.relu(self.enc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar


MOL_LATENT = 10
mol_model = MolecularCVAE(N_DESCRIPTORS, MOL_LATENT).to(device)
mol_opt = torch.optim.Adam(mol_model.parameters(), lr=1e-3)

print(f"Training CVAE on {N_MOLECULES} molecules ({N_DESCRIPTORS} descriptors)...")
for epoch in range(80):
    mol_model.train()
    for (batch,) in mol_loader:
        recon, mu, logvar = mol_model(batch)
        recon_l = F.mse_loss(recon, batch, reduction="sum")
        kl_l = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        jacob = sum(
            torch.sum(p**2) for p in [mol_model.enc1.weight, mol_model.enc2.weight]
        )
        loss = recon_l + kl_l + 1e-4 * jacob
        mol_opt.zero_grad()
        loss.backward()
        mol_opt.step()

# --- Encode all molecules ---
mol_model.eval()
with torch.no_grad():
    all_mu, _ = mol_model.encode(mol_tensor)
    mol_latents = all_mu.cpu().numpy()

# --- Similarity search ---
query_idx = 42
query_latent = mol_latents[query_idx : query_idx + 1]
dists = cdist(query_latent, mol_latents, metric="euclidean")[0]
dists[query_idx] = float("inf")
top_k = 10
nearest = np.argsort(dists)[:top_k]

query_family = family_labels[query_idx]
retrieved_families = family_labels[nearest]
same_family_pct = (retrieved_families == query_family).mean() * 100

print(f"\nQuery molecule #{query_idx} (family: {FAMILY_NAMES[query_family]})")
print(f"Top-{top_k} nearest neighbours:")
for rank, ni in enumerate(nearest):
    fam = FAMILY_NAMES[family_labels[ni]]
    match = "SAME" if family_labels[ni] == query_family else "diff"
    print(f"  #{rank+1}: molecule {ni}, family={fam}, dist={dists[ni]:.3f} [{match}]")
print(f"Same-family retrieval: {same_family_pct:.0f}%")

# --- Visualise latent space (PCA) ---
from numpy.linalg import svd

centered = mol_latents - mol_latents.mean(axis=0)
_, _, Vt = svd(centered, full_matrices=False)
proj = centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(10, 8))
for fam in range(N_FAMILIES):
    mask = family_labels == fam
    ax.scatter(proj[mask, 0], proj[mask, 1], s=10, alpha=0.5, label=FAMILY_NAMES[fam])
ax.scatter(
    proj[query_idx, 0],
    proj[query_idx, 1],
    s=200,
    c="red",
    marker="*",
    zorder=5,
    label="Query",
)
for ni in nearest[:5]:
    ax.annotate(f"#{ni}", (proj[ni, 0], proj[ni, 1]), fontsize=8, color="red")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(
    "Molecular Latent Space (CVAE)\nSmooth clustering enables similarity search",
    fontsize=13,
)
ax.legend(fontsize=9, markerscale=3, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ex1_molecular_latent_space.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Business Impact ---
print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Drug Molecule Similarity Search")
print("=" * 64)
print(f"\nMolecular library: {N_MOLECULES:,} compounds, {N_DESCRIPTORS} descriptors")
print(f"CVAE latent dimension: {MOL_LATENT}")
print(f"Same-family retrieval rate: {same_family_pct:.0f}% (top-{top_k})")
print(f"\nDrug discovery impact:")
print(f"  Traditional screening: test 10,000 compounds at S$100 each = S$1M")
print(f"  CVAE-guided search: prioritise top-100 neighbours first")
print(f"  Expected hit rate improvement: ~3-5x (from random screening)")
print(f"  Cost savings per drug programme: S$200K-400K in early screening")
print(f"\nSmooth latent space advantage:")
print(f"  Interpolation between a hit and a miss suggests optimisation direction")
print(f"  'Move 20% toward molecule X in latent space' = specific structural changes")
print(f"  This is medicinal chemistry guidance from the model itself")
print("=" * 64)



## REFLECTION


[x] Built a Contractive VAE (VAE + Jacobian weight penalty)
  [x] Compared interpolation smoothness: vanilla VAE vs CVAE
  [x] Observed smoother transitions in CVAE's latent space
  [x] Applied to molecular similarity search in drug discovery
  [x] Built latent-space nearest-neighbour retrieval for molecules
  [x] Quantified drug screening cost savings

  KEY INSIGHT: The CVAE is the best of both worlds. The VAE component
  gives you a generative model (sample new molecules). The contractive
  component gives you a smooth latent space (nearby molecules are truly
  similar). Together, they enable both GENERATION (propose new drug
  candidates) and NAVIGATION (find similar molecules, interpolate
  between hits and misses).

  Next: 11_grand_comparison.py puts all 10 variants side by side...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

